<a href="https://colab.research.google.com/github/N-Acil/2022-21-03-Birmingham/blob/gh-pages/Gridding_Africa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cartopy
!pip install matplotlib-scalebar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 55.9 MB/s eta 0:00:00


In [ ]:
import ee
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.patches as mpatches
from google.colab import drive

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-nacil')

In [ ]:
# GEE
drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='ee-nacil')

NameError: name 'drive' is not defined

In [ ]:
UTM = ee.FeatureCollection("users/NXA807/TreeMort/Global/GFC_UTM_Grid_Zones")

grids_to_keep = ["28N","28P","28Q","28R","29N","29P","29Q","29R","29S","30N",
                 "30P","30R","30S","31N","31P","31Q","31S","32M","32N","32P",
                 "32S","33H","33J","33K","33L","33M","33N","33P","33R","33S",
                 "34H","34J","34K","34L","34M","34N","34P","34S","35H","35J",
                 "35K","35L","35M","35N","35P","35R","36J","36K","36L","36M",
                 "36N","36P","36Q","36R","37K","37L","37M","37N","37P","37Q",
                 "38J","38K","38L","38M","38N","38P","39K","39L","39P"]

filtered_UTM = UTM.filter(ee.Filter.inList('ZoneName', grids_to_keep))


def extract_geometry_and_label(feature):
    geom = feature.geometry().coordinates().getInfo()
    coords = []
    for ring in geom[0]:
        coords.append((ring[0], ring[1]))
    # Get center of geometry
    centroid = feature.geometry().centroid().coordinates().getInfo()
    label = feature.get('ZoneName').getInfo()
    return coords, centroid, label

# Extract all geometries and labels
features = filtered_UTM.toList(filtered_UTM.size())
geometries = [extract_geometry_and_label(ee.Feature(features.get(i))) for i in range(features.size().getInfo())]




In [ ]:
# Plotting
fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='white')

# Plot each polygon and label
for coords, centroid, label in geometries:
    xs, ys = zip(*coords)
    ax.fill(xs, ys, alpha=0.25, edgecolor='black',facecolor='#4672C4', linewidth=1)#facecolor='none',
    ax.text(centroid[0], centroid[1], label, ha='center', va='center', fontsize=8, transform=ccrs.PlateCarree())
plt.show()